# DSFB-Database — Paper Reproduction Notebook

This notebook regenerates every figure and table in the
**dsfb-database** paper ("A Deterministic, Read-Only Structural Observer
for Residual Trajectories in SQL Database Telemetry").

## Non-claims (read first)

1. DSFB-Database does not optimise queries, replace the query optimiser, or modify execution plans.
2. DSFB-Database does not claim causal correctness; motifs represent structural consistency given observed signals, not root causes.
3. DSFB-Database does not provide a forecasting or predictive guarantee; precursor structure is structural, not probabilistic.
4. DSFB-Database does not provide ground-truth-validated detection on real workloads; we evaluate via injected perturbations, plan-hash concordance, and replay determinism.
5. DSFB-Database does not claim a universal SQL grammar; motifs are engine-aware, telemetry-aware, and workload-aware.

## Reproducibility contract

* The TPC-DS-shaped perturbation harness runs **fully in-process** —
  no external dataset download is required for the paper's controlled
  results.
* Real-data sections (Snowset, SQLShare, CEB, JOB) require the
  per-dataset fetch scripts; this notebook documents the commands but
  does not auto-run them in Colab.
* Two SHA256 fingerprints are pinned (residual stream + episode stream)
  and verified by the test suite. If either fails, the notebook stops.

## 1. Install Rust toolchain (Colab) and clone the repository

Skip both cells locally if you already have the repo and `cargo` on PATH.

In [ ]:
import shutil, subprocess, os, sys
if shutil.which('cargo') is None:
    subprocess.check_call(['bash', '-lc',
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable"])
    os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
subprocess.check_call(['cargo', '--version'])

In [ ]:
if not os.path.exists('dsfb'):
    subprocess.check_call(['git', 'clone', 'https://github.com/infinityabundance/dsfb.git'])
REPO = os.path.abspath('dsfb')
CRATE = os.path.join(REPO, 'crates', 'dsfb-database')
print('crate path:', CRATE)

## 2. Build + run the test suite

All 15 tests must pass. The pinned-fingerprint tests verify bytewise replay determinism.

In [ ]:
subprocess.check_call(['cargo', 'build', '--release'], cwd=CRATE)
subprocess.check_call(['cargo', 'test', '--release'], cwd=CRATE)

## 3. Run the controlled-perturbation pipeline (Section 8 of the paper)

This is the headline result: F1, TTD, FAR, episode compression, replay
determinism, and threshold elasticity for all five motif classes.

In [ ]:
OUT = os.path.join(CRATE, 'out')
os.makedirs(OUT, exist_ok=True)
subprocess.check_call(['./target/release/dsfb-database', 'reproduce',
                       '--seed', '42', '--out', OUT], cwd=CRATE)
subprocess.check_call(['./target/release/dsfb-database', 'replay-check',
                       '--seed', '42'], cwd=CRATE)
subprocess.check_call(['./target/release/dsfb-database', 'elasticity',
                       '--seed', '42', '--out', OUT], cwd=CRATE)

In [ ]:
import pandas as pd
metrics = pd.read_csv(os.path.join(OUT, 'tpcds.metrics.csv'))
metrics

## 4. Inspect the emitted episode stream

Every row is a closed-form episode `(motif, channel, t_start, t_end, peak, ema_at_boundary, trust_sum)`.
The replay-check step above guarantees this CSV is bytewise identical across runs.

In [ ]:
episodes = pd.read_csv(os.path.join(OUT, 'tpcds.episodes.csv'))
episodes

## 5. Per-motif residual overlay figures

Each PNG shows the residual stream of one motif class with episode
boundaries overlaid. These are the figures embedded in §8.

In [ ]:
from IPython.display import Image, display
for m in ['plan_regression_onset', 'cardinality_mismatch_regime',
         'contention_ramp', 'cache_collapse', 'workload_phase_transition']:
    path = os.path.join(OUT, f'tpcds.{m}.png')
    if os.path.exists(path):
        print(m)
        display(Image(path))

## 6. (Optional) Real-data adapters

These commands fetch and process the public datasets used in §6. They
require network access and ~1 GB of free disk; skip in Colab if the
free tier is too constrained.

```bash
./scripts/fetch_snowset_subset.sh && ./target/release/dsfb-database run --dataset snowset  --path data/snowset_shard.csv --out out
./scripts/fetch_sqlshare.sh        && ./target/release/dsfb-database run --dataset sqlshare --path data/sqlshare_queries.csv --out out
./scripts/fetch_ceb.sh             && ./target/release/dsfb-database run --dataset ceb      --path data/ceb_subset.csv --out out
./scripts/fetch_job.sh             && ./target/release/dsfb-database run --dataset job      --path data/job_manifest.csv --out out
./scripts/build_tpcds.sh           && ./target/release/dsfb-database run --dataset tpcds    --path data/tpcds_trace.csv --out out
```

## 7. (Optional) Build the paper PDF

Requires `latexmk`. The `paper/build.sh` script handles the rest.

```bash
cd paper && bash build.sh
```

On Colab: `apt-get install -y texlive-latex-extra latexmk` first.